# Задание 5.
## Исходные данные

Для анализа были использованы данные секвенирования из проекта PRJEB84057. В работе рассматривались четыре FASTQ-файла.

Список использованных файлов приведён в файле:

`data_files_used.txt`



## Часть 1. Анализ качества сырых данных FastQC/MultiQC

Для первичной оценки качества секвенирования использовалась программа FastQC. Полученные отчёты были объединены с помощью MultiQC.

MultiQC-отчёт до тримминга находится по пути:

`results/fastqc_raw/multiqc_report.html`

Ниже приведены основные скриншоты отчёта, отражающие качество исходных данных.

### Скриншоты MultiQC до тримминга



![Part 1 screenshot 1](screenshots/part_1/part_1_1.png)



![Part 1 screenshot 2](screenshots/part_1/part_1_2.png)



![Part 1 screenshot 3](screenshots/part_1/part_1_3.png)


![Part 1 screenshot 4](screenshots/part_1/part_1_4.png)

### SLURM-скрипт для FastQC и MultiQC до тримминга

```
#!/bin/bash
#SBATCH --job-name=fastqc_raw
#SBATCH --output=logs/fastqc_raw_%j.out
#SBATCH --error=logs/fastqc_raw_%j.err
#SBATCH --time=01:00:00
#SBATCH --cpus-per-task=4
#SBATCH --mem=8G
#SBATCH --export=NONE 
#SBATCH -p AMD9554-common

CONDA_BASE=$(conda info --base)
source $CONDA_BASE/etc/profile.d/conda.sh
conda activate bioinfo_env

cd ~/hw5/data/raw_fastq || exit 1
fastqc *.fastq.gz -t $SLURM_CPUS_PER_TASK -o ~/hw5/results/fastqc_raw
multiqc ~/hw5/results/fastqc_raw -o ~/hw5/results/fastqc_raw



```

## Часть 2. Тримминг ридов

На основании отчёта MultiQC для исходных данных был выполнен тримминг ридов. Целью этого этапа было удалить возможные адаптерные последовательности, обрезать низкокачественные участки и исключить слишком короткие риды.




### SLURM-скрипт для тримминга

```
#!/bin/bash
#SBATCH --job-name=fastp_trim
#SBATCH --output=logs/fastp_trim_%j.out
#SBATCH --error=logs/fastp_trim_%j.err
#SBATCH --time=01:00:00
#SBATCH --cpus-per-task=4
#SBATCH --mem=8G
#SBATCH -p AMD9554-common

CONDA_BASE=/home/STUDY/FBMF/bioinformatics/anaconda3
source $CONDA_BASE/etc/profile.d/conda.sh || { echo "Conda path not found"; exit 1; }
conda activate bioinfo_env

export PATH="$CONDA_BASE/envs/bioinfo_env/bin:$PATH"

cd ~/hw5/data/raw_fastq || exit 1
mkdir -p ~/results/fastp_reports

for fastq_file in *.fastq.gz; do
    
    base_name=$(basename "$fastq_file" .fastq.gz)
    
    fastp -i "$fastq_file" \
          -o ~/hw5/data/trimmed_fastq/"${base_name}_trimmed.fastq.gz" \
          --detect_adapter_for_pe \
          --cut_right \
	  --cut_window_size 5 \
          --cut_mean_quality 20 \
          --length_required 36 \
          --thread $SLURM_CPUS_PER_TASK \
          --html ~/results/fastp_reports/"${base_name}_fastp_report.html" \
          --json ~/results/fastp_reports/"${base_name}_fastp_report.json"
done


```

### Проверка результатов после тримминга

Очищенные FASTQ-файлы не включались в репозиторий, чтобы не загружать тяжёлые данные. Однако результат тримминга был проверен повторным запуском FastQC и MultiQC по trimmed-файлам.

В папке `fastqc_trimmed` находятся HTML- и ZIP-отчёты FastQC для файлов с суффиксом `_trimmed`, а также итоговый отчёт `multiqc_report.html`.


Результат:

```text
total 13512
-rw-r--r--@  1 mihailgusev  staff  2388611 May 18 21:48:14 2026 multiqc_report.html
-rw-r--r--   1 mihailgusev  staff   472789 May 18 21:20:34 2026 ERR14230600_Illumina_HiSeq_4000_sequencing_trimmed_fastqc.zip
drwxr-xr-x  12 mihailgusev  staff      384 May 18 21:20:34 2026 .
-rw-r--r--   1 mihailgusev  staff   642064 May 18 21:20:34 2026 ERR14230586_Illumina_HiSeq_4000_sequencing_trimmed_fastqc.html
-rw-r--r--   1 mihailgusev  staff   658946 May 18 21:20:34 2026 ERR14230595_Illumina_HiSeq_4000_sequencing_trimmed_fastqc.html
-rw-r--r--   1 mihailgusev  staff   484231 May 18 21:20:34 2026 ERR14230582_Illumina_HiSeq_4000_sequencing_trimmed_fastqc.zip
-rw-r--r--   1 mihailgusev  staff   664341 May 18 21:20:34 2026 ERR14230582_Illumina_HiSeq_4000_sequencing_trimmed_fastqc.html
-rw-r--r--   1 mihailgusev  staff   476189 May 18 21:20:34 2026 ERR14230595_Illumina_HiSeq_4000_sequencing_trimmed_fastqc.zip
-rw-r--r--   1 mihailgusev  staff   652429 May 18 21:20:34 2026 ERR14230600_Illumina_HiSeq_4000_sequencing_trimmed_fastqc.html
drwxr-xr-x  22 mihailgusev  staff      704 May 18 21:20:34 2026 multiqc_data
-rw-r--r--   1 mihailgusev  staff   456083 May 18 21:20:34 2026 ERR14230586_Illumina_HiSeq_4000_sequencing_trimmed_fastqc.zip
drwxr-xr-x   4 mihailgusev  staff      128 May 18 21:20:34 2026 ..
```



## Часть 3. Контроль качества после тримминга

После тримминга был повторно запущен FastQC для очищенных файлов. Затем отчёты FastQC были объединены в общий отчёт MultiQC.

MultiQC-отчёт после тримминга находится по пути:

`results/fastqc_trimmed/multiqc_report.html`


### SLURM-скрипт для FastQC/MultiQC после тримминга

```
#!/bin/bash
#SBATCH --job-name=fastqc_trim
#SBATCH --output=logs/fastqc_trim_%j.out
#SBATCH --error=logs/fastqc_trim_%j.err
#SBATCH --time=01:00:00
#SBATCH --cpus-per-task=4
#SBATCH --mem=8G
#SBATCH --export=NONE
#SBATCH -p AMD9554-common

CONDA_BASE=$(conda info --base)
source $CONDA_BASE/etc/profile.d/conda.sh
conda activate bioinfo_env

cd ~/hw5/data/trimmed_fastq || exit 1
fastqc *.fastq.gz -t $SLURM_CPUS_PER_TASK -o ~/hw5/results/fastqc_trimmed
multiqc ~/hw5/results/fastqc_trimmed -o ~/hw5/results/fastqc_trimmed

```

### Скриншоты MultiQC после тримминга


![MultiQC after trimming 1](screenshots/part_3/part_3_1.png)


![MultiQC after trimming 2](screenshots/part_3/part_3_2.png)


![MultiQC after trimming 3](screenshots/part_3/part_3_3.png)


![MultiQC after trimming 4](screenshots/part_3/part_3_4.png)

## Общий вывод



На первом этапе FastQC и MultiQC позволили оценить основные характеристики исходных данных. На основании этих результатов был выполнен тримминг с использованием fastp.

После тримминга данные были повторно проанализированы с помощью FastQC и MultiQC. По результатам можно сделать вывод, что исходники были качественные. 